## Setup — imports & paths

In [ ]:
import json
import time
from getpass import getpass
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

root = Path.cwd().parent                     # notebook lives in "llm/"
notes_csv = root / "data" / "followup_notes.csv"
clean_csv = root / "appointments_clean.csv"
analysed_csv = root / "notes_analysed.csv"

MODEL_NAME = "gemini-3-flash-preview"        # use the Flash model from your Session 22.1 notebook
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]
PAUSE_SECONDS = 13                           # free tier ~5 requests/minute
api_calls = 0                                # every real API call adds 1 (Task 4.3c)

## Task 4.1 — One validated call

### 4.1.1 API key (input) & client

In [ ]:
API_KEY = getpass("Paste your Gemini API key: ")   # hidden input; nothing is printed
client = genai.Client(api_key=API_KEY)

### 4.1.2 System instruction, CRAFT prompt, JSON schema

In [ ]:
SYSTEM_TEXT = ("You classify why a clinic patient missed an appointment. "
               "Reply only with JSON that matches the given schema.")

def craft_prompt(note_text):
    """V2 prompt: Context, Role, Action, Tone, Format, with the allowed values."""
    return f"""Context: CareLoop Clinics phones patients who missed an appointment. The call-centre agent wrote this note (it may be in English or Hinglish):
\"\"\"{note_text}\"\"\"

Role: You are an operations analyst who reads these notes to find out why patients miss appointments.

Action: Pick the single main reason the patient missed. root_cause must be exactly one of:
{", ".join(ALLOWED)}.
If the note does not clearly say why the patient missed (for example, the patient could not be reached), answer "other" rather than guess.

Tone: Factual and neutral. Do not add details that are not in the note.

Format: JSON with root_cause (one allowed value), confidence (a number from 0 to 1), summary (at most 25 words, in English) and recommended_action (one short step the clinic can take)."""

def v1_prompt(note_text):
    """V1 prompt for Task 4.3(b): no context, no allowed values."""
    return f"Classify this note: {note_text}"

SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"},          # plain string, NOT an enum
                   "confidence": {"type": "number"},
                   "summary": {"type": "string"},
                   "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

### 4.1.3 `analyse_note(note_text)`

In [ ]:
def call_gemini(prompt):
    """Send one prompt and return the reply text exactly as the API returned it."""
    global api_calls
    api_calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),  # delete if your model rejects it
        ),
    )
    return response.text

def analyse_note(note_text, prompt_fn=craft_prompt):
    """Classify one note and return the parsed JSON dictionary."""
    return json.loads(call_gemini(prompt_fn(note_text)))

### 4.1.4 Run on N001

In [ ]:
notes = pd.read_csv(notes_csv)
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]

result_n001 = analyse_note(n001)
print(json.dumps(result_n001, indent=2, ensure_ascii=False))
print("Fields:", sorted(result_n001))

## Task 4.2 — Validate and process the 10 notes

### 4.2(a) Pydantic `FollowupInsight` + allowed-list check

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_reply(data):
    """Pydantic validation + allowed-list check. Raises on any problem."""
    insight = FollowupInsight(**data)
    if insight.root_cause not in ALLOWED:
        raise ValueError(f"root_cause {insight.root_cause!r} is not in the allowed list")
    return insight

### 4.2(b) + (c) Loop over the first 10 notes (blank-note branch, `try/except`, 13 s pause)

The pause is inside the loop, before every call except the first, so the free-tier limit is respected. Blank notes skip both the call and the pause.

In [ ]:
FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
rows = []
first_call = True

for _, note in notes.head(10).iterrows():
    note_text = note["note_text"]
    row = {"note_id": note["note_id"], "appointment_id": note["appointment_id"]}

    if pd.isna(note_text) or str(note_text).strip() == "":   # isna first: NaN has no .strip()
        row.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank",
                   raw_response="")
        rows.append(row)
        print(f"{note['note_id']}: skipped_blank (no API call)")
        continue

    if not first_call:
        time.sleep(PAUSE_SECONDS)
    first_call = False

    raw = ""
    try:
        raw = call_gemini(craft_prompt(note_text))
        data = json.loads(raw)
        row.update({f: data.get(f) for f in FIELDS})   # copy whatever the reply contained, unrepaired
        check_reply(data)
        row["validation_status"] = "valid"
    except Exception as e:                             # one bad reply must not stop the loop
        row["validation_status"] = "invalid"
        print(f"{note['note_id']}: invalid — {type(e).__name__}: {e}")
    row["raw_response"] = raw                          # exactly as returned
    rows.append(row)
    print(f"{note['note_id']}: {row.get('root_cause')} ({row['validation_status']})")

### 4.2(d) Save `notes_analysed.csv`

In [ ]:
columns = ["note_id", "appointment_id", "root_cause", "confidence", "summary",
           "recommended_action", "validation_status", "raw_response"]
analysed = pd.DataFrame(rows).reindex(columns=columns)   # missing fields stay empty
analysed.to_csv(analysed_csv, index=False)

print(f"Saved {analysed_csv.name}: {analysed.shape}")
print(analysed["validation_status"].value_counts().to_string())
analysed[["note_id", "root_cause", "confidence", "validation_status"]]

## Task 4.3 — Break it on purpose

### 4.3(a) Two deliberately bad replies

In [ ]:
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Fee felt too high this month.",
                         "recommended_action": "Offer the follow-up discount."},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Fee felt too high this month."},
}
for name, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(f"{name}: unexpectedly passed")
    except ValidationError as e:
        print(f"--- {name} ---")
        for err in e.errors():
            print(f"  field {err['loc']}: {err['msg']} (type: {err['type']})")

### 4.3(b) V1 vs V2 prompt on N007 & N009

Same schema and settings for both (plain-string `root_cause`, JSON mode, `temperature=0`); only the prompt differs.

In [ ]:
compare = []
for note_id in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]
    for version, prompt_fn in [("V1 (just classify)", v1_prompt), ("V2 (CRAFT)", craft_prompt)]:
        time.sleep(PAUSE_SECONDS)
        try:
            cause = analyse_note(text, prompt_fn).get("root_cause")
        except Exception as e:
            cause = f"error: {type(e).__name__}"
        compare.append({"note": note_id, "prompt": version, "root_cause": cause,
                        "allowed?": cause in ALLOWED})

pd.DataFrame(compare)

**Answer:** _(fill in after your run, from the table above)_ V2 returns an allowed value for both notes, while V1 makes up its own labels because it never sees the allowed list. I would deploy **V2**: its output can be counted and acted on, and it answers `other` for the ambiguous N009 instead of guessing a reason the note does not give.

### 4.3(c) Total API calls

In [ ]:
print(f"API calls used in Tasks 4.1–4.3: {api_calls}  (1 in 4.1 + 9 in 4.2 + 4 in 4.3b)")
assert api_calls < 25, "over the free-tier budget"

## Task 4.4 — Recommendation and GitHub

### 4.4(a) Merge & crosstab `root_cause` × `reminder_sent`

In [ ]:
appointments = pd.read_csv(clean_csv)
merged = pd.read_csv(analysed_csv).merge(appointments[["appointment_id", "reminder_sent"]],
                                          on="appointment_id", how="left")
print(f"Merged rows: {len(merged)}")
print(f"Notes on appointments with reminder_sent = No: {(merged['reminder_sent'] == 'No').sum()} of {len(merged)}")
pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)

**Answer:** _(check against your crosstab)_ Both notes classed `forgot_or_no_reminder` (N001, N007) belong to appointments with **no reminder** (2 of 2), and 5 of the 10 notes are on appointments with `reminder_sent = No`. This agrees with Part 3, where `reminder_sent` had the most negative coefficient (odds about 69% lower with a reminder). With only 10 notes this is an illustration, not proof of the clinic-wide reasons. The full recommendation is in `README.md`.